# CS781 Phase 4 — Notebook P4_02: Court Judgment Prediction with the Phase-1 models

Binary outcome (1 = appeal accepted) from a long judgment. One notebook runs a queue of experiments (one per GPU)
and writes, for each, dev scores (official macro-F1, threshold calibrated on dev), an FP/FN analysis and a ready
`submission.zip`.

| Step | What happens |
|---|---|
| 1 | config: Phase-1 weights, experiments, queue |
| 2 | inputs: P4_01's token files (memory-mapped) + Phase-1 checkpoints |
| 3 | model classes (NB3) + backbone and adaptation methods (P3_02) |
| 4 | **document representations**: head / middle / tail / head+tail of L tokens, or the last K chunks of C tokens |
| 5 | document model: token pooling (mean / end token / attention) → chunk pooling (attention / mean / BiGRU) → classifier |
| 6 | techniques: full, frozen, top-k, **gradual unfreezing**, BitFit, **LoRMA**, adapters (+ LoRA, IA³); smaller students (`keep_layers`) |
| 7 | training (+ knowledge distillation), dev threshold calibration, FP/FN analysis, submission |
| 8 | **hierarchical two-stage model**: a trained encoder embeds up to K chunks per document once → BiGRU + attention over chunks |
| 9 | unit tests | 10 | run the queue | 11 | results table |

**Kaggle settings**: Accelerator **GPU T4 x2**, Internet not needed.
**Inputs**: (1) the **P4_01 output** (`p4/data/…`), (2) the **Phase-1 weight datasets**, (3) *optional* earlier
versions of this notebook (finished experiments are skipped; KD teachers / hierarchical encoders are found there).

In [1]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
  disk @ /tmp/cs781_scratch: free 1148.8 GB / total 8758.4 GB
HF token found: True


## 1. Configuration

In [2]:
import hashlib, torch, torch.nn as nn, torch.nn.functional as F, zipfile, threading, queue as _queue, traceback, itertools
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
try: display
except NameError: display = print

P4_SMOKE = os.environ.get("CS781_P4_SMOKE") == "1"

P1_SOURCES = {
    "A0": dict(dataset="cs781-p1-model-a0-sc80", run_name="A0_gpt2_110M_SC80_epoch_1000M"),
    "A1": dict(dataset="cs781-p1-model-a1-sc80", run_name="A1_modern_110M_SC80_epoch_1000M"),
    "B":  dict(dataset="cs781-p1-model-b", run_name="B_encdec_6x6_SC80_epoch_1000M"),
    "E":  dict(dataset="data-partial-e-prefixlm-110m-sc80-epoch-1000m", run_name="E_prefixlm_110M_SC80_epoch_1000M"),
    "F":  dict(dataset="cs781-p1-model-f", run_name="F_localglobal_gqa_alibi_SC80_epoch_1000M"),
}

DEFAULTS = dict(
    model="E", attn="bidir", init="pretrained", keep_layers=None,       # keep_layers=k: only the first k blocks (smaller student)
    method="full", lr=None,
    lora_r=16, lora_alpha=32, lora_targets="all", lorma_variant="plus", topk=4, adapter_dim=64, prompt_len=0,
    # ---- document representation ----
    rep="tail",           # "tail" | "head" | "middle" | "headtail" (first head_tokens + last rest) | "chunks" (last n_chunks x chunk_len)
    seq_len=512, head_tokens=128, n_chunks=4, chunk_len=512,
    tok_pool="mean",      # "mean" | "last" (state at the end separator — the summary position for causal models) | "attn"
    chunk_pool="attn",    # how chunk vectors are combined (rep="chunks"): "attn" | "mean" | "gru"
    head_dropout=0.1, backbone_dropout=0.1,
    # ---- data / optimisation ----
    max_train_docs=None,  # e.g. 10000: random (stratified) subset of the training documents for cheaper ablations
    epochs=2, batch=8, accum=2, lr_head=1e-3, weight_decay=0.01, warmup=0.06, grad_clip=1.0,
    class_weight="balanced",   # None | "balanced" (inverse class frequency of the training docs)
    teacher=None, kd_alpha=0.5, kd_T=2.0,
    # ---- hierarchical two-stage ----
    hier_from=None, hier_chunks=8, hier_epochs=12, hier_max_train_docs=None,
    seed=0,
)
METHOD_LR = {"full": 3e-5, "frozen": 0.0, "topk": 5e-5, "gradual": 3e-5, "bitfit": 1e-3, "lorma": 5e-4, "lora": 5e-4,
             "rslora": 5e-4, "adapter": 1e-3, "ia3": 3e-3}

SUB = dict(max_train_docs=10000, epochs=3)            # ablation set-up: 10k training docs
EXPERIMENTS = {
    # ---- Stage 1: which part of the judgment / which long-document representation (E, full FT, 10k docs) ----
    "s1_E_head512":       dict(rep="head", **SUB),
    "s1_E_middle512":     dict(rep="middle", **SUB),
    "s1_E_tail512":       dict(rep="tail", **SUB),
    "s1_E_headtail512":   dict(rep="headtail", **SUB),
    "s1_E_tail1024":      dict(rep="tail", seq_len=1024, batch=4, accum=4, **SUB),
    "s1_E_chunks4_attn":  dict(rep="chunks", n_chunks=4, batch=2, accum=8, **SUB),
    "s1_E_tail512_attnpool": dict(rep="tail", tok_pool="attn", **SUB),
    "s1_A1_random_init":  dict(model="A1", attn="causal", tok_pool="last", init="random", lr=3e-4, **SUB),
    # ---- Stage 2: Phase-1 backbones (all training docs, tail 512) ----
    "s2_A0_causal":       dict(model="A0", attn="causal", tok_pool="last"),
    "s2_A1_causal":       dict(model="A1", attn="causal", tok_pool="last"),
    "s2_E":               dict(model="E"),
    "s2_F_causal":        dict(model="F", attn="causal", tok_pool="last"),
    "s2_B_encoder":       dict(model="B"),
    # ---- Stage 3: adaptation techniques (E, all docs, tail 512) ----
    "s3_E_frozen":        dict(method="frozen", epochs=3),
    "s3_E_top4":          dict(method="topk", topk=4),
    "s3_E_gradual":       dict(method="gradual", epochs=3),
    "s3_E_bitfit":        dict(method="bitfit", epochs=3),
    "s3_E_lorma":         dict(method="lorma", epochs=3),
    "s3_E_lorma_pi":      dict(method="lorma", lorma_variant="pi", lora_targets="attn", epochs=3),
    "s3_E_adapter":       dict(method="adapter", epochs=3),
    # ---- Stage 4 (needs the stage-2 run attached): hierarchical models, distillation, LoRMA on other backbones ----
    "s4_hier8_E":         dict(hier_from="s2_E", hier_chunks=8),
    "s4_hier8_A1":        dict(hier_from="s2_A1_causal", hier_chunks=8),
    "s4_hier8_F":         dict(hier_from="s2_F_causal", hier_chunks=8),
    "s4_hier8_A0":        dict(hier_from="s2_A0_causal", hier_chunks=8),
    "s4_hier16_E":        dict(hier_from="s2_E", hier_chunks=16),          # 16 x 512 = 8192 tokens: whole document for ~89% of cases
    "s4_E_chunks4_all":   dict(rep="chunks", n_chunks=4, batch=2, accum=8),   # best stage-1 representation, all training docs
    "s4_A1_6L_control":   dict(model="A1", attn="causal", tok_pool="last", keep_layers=6),
    "s4_A1_6L_kd":        dict(model="A1", attn="causal", tok_pool="last", keep_layers=6, teacher="s2_E"),
    "s4_B_kd":            dict(model="B", teacher="s2_E"),
    "s4_F_lorma":         dict(model="F", attn="causal", tok_pool="last", method="lorma", epochs=3),
    "s4_A0_lorma":        dict(model="A0", attn="causal", tok_pool="last", method="lorma", epochs=3),
}
QUEUES = {f"stage{i}": [k for k in EXPERIMENTS if k.startswith(f"s{i}_")] for i in (1, 2, 3, 4)}
RUN_QUEUE = QUEUES["stage3"]
MIN_HOURS_PER_EXP = 0.75

if P4_SMOKE:
    P1_SOURCES = {k: dict(dataset="", run_name=n) for k, n in {
        "A0": "SMOKE_A0_gpt2_110M_mix50_300M", "A1": "SMOKE_A1_modern_110M_mix50_300M",
        "B": "SMOKE_B_encdec_6x6_mix50_epoch_300M", "E": "SMOKE_E_prefixlm_110M_mix50_epoch_300M",
        "F": "SMOKE_F_localglobal_gqa_alibi_mix50_epoch_300M"}.items()}
    DEFAULTS.update(epochs=1, seq_len=128, chunk_len=128, max_train_docs=400, hier_epochs=2, hier_chunks=3)
    for k in EXPERIMENTS:
        EXPERIMENTS[k].update(epochs=1, max_train_docs=400)
        if EXPERIMENTS[k].get("seq_len"): EXPERIMENTS[k]["seq_len"] = 256
        if EXPERIMENTS[k].get("keep_layers"): EXPERIMENTS[k]["keep_layers"] = 1
    RUN_QUEUE = [x for x in os.environ.get("CS781_P4_QUEUE", "s1_E_tail512").split(",") if x]

def exp_cfg(name):
    cfg = dict(DEFAULTS); cfg.update(EXPERIMENTS[name]); cfg["name"] = name
    if cfg["lr"] is None: cfg["lr"] = METHOD_LR[cfg["method"]]
    return cfg

EXPS_DIR = ROOT / "p4" / "exps"; EXPS_DIR.mkdir(parents=True, exist_ok=True)
NUM_GPUS = torch.cuda.device_count()
print("GPUs:", NUM_GPUS, "| queue:", RUN_QUEUE)

GPUs: 2 | queue: ['s3_E_frozen', 's3_E_top4', 's3_E_gradual', 's3_E_bitfit', 's3_E_lorma', 's3_E_lorma_pi', 's3_E_adapter']


## 2. Inputs: token files (memory-mapped) and Phase-1 checkpoints

In [3]:
class Split:
    def __init__(self, name):
        self.name = name
        self.tok = np.memmap(find_artifact(f"p4/data/{name}_tokens.bin"), dtype=np.uint16, mode="r")
        off = np.load(find_artifact(f"p4/data/{name}_offsets.npy")); self.start, self.end = off[:-1], off[1:]
        m = load_json(find_artifact(f"p4/data/{name}_meta.json"))
        self.ids, self.labels = m["ids"], (np.asarray(m["labels"], dtype=np.int64) if m["labels"] is not None else None)
        self.lengths = self.end - self.start
    def __len__(self): return len(self.ids)
    def tokens(self, i): return np.asarray(self.tok[self.start[i]:self.end[i]], dtype=np.int64)
    def doc_hash(self, i): return hashlib.blake2b(np.asarray(self.tok[self.start[i]:self.end[i]]).tobytes(), digest_size=8).digest()
    def subset(self, keep):
        s = Split.__new__(Split); s.name, s.tok = self.name, self.tok; keep = np.asarray(keep)
        s.start, s.end, s.lengths = self.start[keep], self.end[keep], self.lengths[keep]
        s.ids = [self.ids[i] for i in keep]; s.labels = self.labels[keep] if self.labels is not None else None
        return s

assert find_artifact("p4/data/train_tokens.bin") is not None, "attach the P4_01 output (p4/data/…)"
SPLITS = {s: Split(s) for s in ("train", "dev", "test")}

# The organisers' dev file contains every test document (P4_01: "dev ∩ test identical docs: 1517"), WITH its label.
# Selecting epochs / thresholds / models on those documents would be tuning on the test set, so dev is restricted to the
# documents that are NOT in test ("clean dev"). Nothing is ever trained on dev.
DEV_EXCLUDE_TEST_COPIES = True
_test_h = {SPLITS["test"].doc_hash(i) for i in range(len(SPLITS["test"]))}
_dup = np.array([SPLITS["dev"].doc_hash(i) in _test_h for i in range(len(SPLITS["dev"]))], bool)
print(f"dev documents identical to a test document: {_dup.sum()} of {len(_dup)}")
if DEV_EXCLUDE_TEST_COPIES and _dup.any():
    SPLITS["dev"] = SPLITS["dev"].subset(np.where(~_dup)[0]); print(f"=> clean dev = {len(SPLITS['dev'])} documents")
for s, sp in SPLITS.items():
    print(f"{s}: {len(sp)} docs | tokens/doc mean {sp.lengths.mean():.0f}, median {np.median(sp.lengths):.0f} | "
          + (f"accepted {sp.labels.mean():.1%}" if sp.labels is not None else "no labels"))
SEP_ID, PAD_ID = 0, 1

if IN_KAGGLE:
    n = 0
    for cand in glob.glob(f"/kaggle/input/**/{PROJECT}/p4/exps/*", recursive=True):
        for src in Path(cand).glob("*"):
            if src.suffix in (".json", ".zip", ".txt"):
                dst = EXPS_DIR / Path(cand).name / src.name
                if not dst.exists(): dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
    print("restored", n, "result files from attached earlier runs")

def find_p1_dir(key):
    src = P1_SOURCES[key]
    roots = [os.environ["CS781_P1_RUNS"]] if os.environ.get("CS781_P1_RUNS") else []
    roots += [str(ROOT / "runs")] + (["/kaggle/input"] if IN_KAGGLE else [])
    cands = sorted({c for r in roots for c in glob.glob(f"{r}/**/weights_final.pt", recursive=True)})
    by_run = [c for c in cands if Path(c).parent.name == src["run_name"]]
    by_ds = [c for c in cands if src["dataset"] and src["dataset"].lower() in [p.lower() for p in Path(c).parts]]
    hit = (by_run or by_ds or [None])[0]
    return Path(hit).parent if hit else None
P1_DIRS = {k: find_p1_dir(k) for k in P1_SOURCES}
for k, p in P1_DIRS.items(): print(f"  {k}: {p if p else 'NOT FOUND'}")

dev documents identical to a test document: 1517 of 2511
=> clean dev = 994 documents
train: 31652 docs | tokens/doc mean 3889, median 2508 | accepted 41.4%
dev: 994 docs | tokens/doc mean 4476, median 3393 | accepted 50.0%
test: 1517 docs | tokens/doc mean 4595, median 3499 | no labels
restored 30 result files from attached earlier runs
  A0: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a0-sc80/cs781_p1/runs/A0_gpt2_110M_SC80_epoch_1000M
  A1: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a1-sc80/cs781_p1/runs/A1_modern_110M_SC80_epoch_1000M
  B: /kaggle/input/datasets/aaravoswal/cs781-p1-model-b/cs781_p1/runs/B_encdec_6x6_SC80_epoch_1000M
  E: /kaggle/input/datasets/chrishitha2403/data-partial-e-prefixlm-110m-sc80-epoch-1000m/cs781_p1/runs/E_prefixlm_110M_SC80_epoch_1000M
  F: /kaggle/input/datasets/chrishitha2403/cs781-p1-model-f/cs781_p1/runs/F_localglobal_gqa_alibi_SC80_epoch_1000M


## 3. Model classes (NB3) and backbone + adaptation methods (P3_02)

In [4]:
# (model classes copied unchanged from NB3's model cell)
class CausalSelfAttention(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.h, self.hd = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.proj.RESIDUAL_SCALE = True
        self.dropout = dropout

    def forward(self, x, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, self.hd).transpose(1, 2) for t in (q, k, v)]
        if past is not None:
            k = torch.cat([past[0], k], dim=2); v = torch.cat([past[1], v], dim=2)
        present = (k, v) if use_cache else None
        drop = self.dropout if self.training else 0.0
        if mask is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=drop)
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop)
        return self.proj(y.transpose(1, 2).reshape(B, T, C)), present

class MLP(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.proj.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class Block(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn, self.mlp = CausalSelfAttention(d, h, dropout), MLP(d, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.ln1(x), past, mask, use_cache)
        x = x + self.drop(a)
        x = x + self.mlp(self.ln2(x))
        return x, present

class GPT2LM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte, self.wpe = nn.Embedding(vocab, d_model), nn.Embedding(ctx_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight                     # weight tying
        for name, m in self.named_modules():
            if isinstance(m, nn.Linear):
                std = 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02
                nn.init.normal_(m.weight, 0.0, std)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.drop(self.wte(idx) + self.wpe(pos))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.ln_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ A1: modern decoder
class RMSNorm(nn.Module):
    """x / sqrt(mean(x²)) · g — no mean subtraction, no bias (cheaper than LayerNorm, equally stable). Computed in fp32."""
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight).to(x.dtype)

def rope_tables(head_dim, n_pos, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    ang = torch.outer(torch.arange(n_pos).float(), inv)                    # (n_pos, head_dim/2)
    return torch.cos(ang), torch.sin(ang)

def apply_rope(x, cos, sin):
    """Rotate each (x1_i, x2_i) pair of the head dimension by angle pos·θ_i. The dot product q·k then depends only on
    the RELATIVE distance between the two positions. x: (B, H, T, D); cos/sin: (B or 1, 1, T, D/2)."""
    d = x.size(-1) // 2
    x1, x2 = x[..., :d].float(), x[..., d:].float()
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).to(x.dtype)

class ModernAttention(nn.Module):
    def __init__(self, d, h, kv, dropout, rope, causal=True):
        super().__init__()
        self.h, self.kv, self.hd, self.rope, self.dropout, self.causal = h, kv, d // h, rope, dropout, causal
        self.q = nn.Linear(d, d, bias=False)
        self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)             # GQA when kv < h: fewer key/value heads
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if self.rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        drop = self.dropout if self.training else 0.0
        # no explicit mask: causal for decoders (but a single cached query may see every past key), full for encoders
        y = (F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and past is None, dropout_p=drop)
             if mask is None else F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class SwiGLU(nn.Module):
    """out = W2( SiLU(W1 x) ⊙ W3 x ): a learned gate decides, per hidden unit, how much of W3 x passes through."""
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False); self.w2.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class GeluMLP(nn.Module):
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False)
        self.proj.RESIDUAL_SCALE = True; self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class ModernBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, norm, mlp, rope):
        super().__init__()
        Norm = RMSNorm if norm == "rms" else nn.LayerNorm
        self.n1, self.n2 = Norm(d), Norm(d)
        self.attn = ModernAttention(d, h, kv, dropout, rope)
        self.mlp = SwiGLU(d, hidden, dropout) if mlp == "swiglu" else GeluMLP(d, hidden, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.n1(x), cos, sin, past, mask, use_cache)
        x = x + self.drop(a)
        return x + self.mlp(self.n2(x)), present

class ModernLM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=None, ffn_hidden=2048,
                 rope_base=10000, pos="rope", norm="rms", mlp="swiglu", **_):
        super().__init__()
        self.ctx_len, self.use_rope = ctx_len, pos == "rope"
        self.wte = nn.Embedding(vocab, d_model)
        self.wpe = None if self.use_rope else nn.Embedding(ctx_len, d_model)
        hd = d_model // n_head
        cos, sin = rope_tables(hd, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, norm, mlp,
                                                 self.use_rope) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model) if norm == "rms" else nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.wte(idx)
        if self.wpe is not None: x = x + self.wpe(pos)
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        x = self.drop(x)
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cos, sin, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ B: encoder-decoder
class CrossAttention(nn.Module):
    """decoder queries attend to the encoder's output ("memory"); keys/values are computed once per prompt"""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, mem, mem_valid=None, kv=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        if kv is None:
            S = mem.size(1)
            kv = tuple(self.kvp(mem).view(B, S, 2, self.h, self.hd).permute(2, 0, 3, 1, 4))
        k, v = kv
        mask = None if mem_valid is None else mem_valid[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o(y.transpose(1, 2).reshape(B, T, C)), kv

class EncoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = ModernAttention(d, h, h, dropout, rope=True, causal=False)     # bidirectional
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mask=None):
        x = x + self.attn(self.n1(x), cos, sin, None, mask, False)[0]
        return x + self.mlp(self.n2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2, self.n3 = RMSNorm(d), RMSNorm(d), RMSNorm(d)
        self.self_attn = ModernAttention(d, h, h, dropout, rope=True, causal=True)
        self.cross = CrossAttention(d, h)
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mem, mem_valid, cache=None, use_cache=False):
        a, self_kv = self.self_attn(self.n1(x), cos, sin, cache["self"] if cache else None, None, use_cache)
        x = x + a
        c, cross_kv = self.cross(self.n2(x), mem, mem_valid, cache["cross"] if cache else None)
        x = x + c
        return x + self.mlp(self.n3(x)), ({"self": self_kv, "cross": cross_kv} if use_cache else None)

class EncDecRunner:
    """generation for the encoder-decoder: encode the prompt once, then decode from <|bos|> one token at a time"""
    def __init__(self, model, rows, pad_id, device, amp):
        self.model, self.device, self.amp = model, device, amp
        B, L = len(rows), max(len(r) for r in rows)
        idx = torch.full((B, L), pad_id, dtype=torch.long); valid = torch.zeros((B, L), dtype=torch.bool)
        for r, ids_ in enumerate(rows):                                        # right padding (bidirectional encoder)
            idx[r, :len(ids_)] = torch.tensor(ids_); valid[r, :len(ids_)] = True
        self.idx, self.mem_valid = idx.to(device), valid.to(device)
    def prefill(self):
        with self.amp:
            self.mem = self.model.encode(self.idx, self.mem_valid)
            bos = torch.full((self.idx.size(0), 1), self.model.bos_id, dtype=torch.long, device=self.device)
            logits, self.cache = self.model.decode(bos, self.mem, self.mem_valid, 0, None, use_cache=True)
        self.t = 0
        return logits[:, -1].float()
    def step(self, nxt):
        self.t += 1
        with self.amp:
            logits, self.cache = self.model.decode(nxt[:, None], self.mem, self.mem_valid, self.t, self.cache, use_cache=True)
        return logits[:, -1].float()
    def reorder(self, idx):
        self.mem, self.mem_valid = self.mem[idx], self.mem_valid[idx]; self.cache = reorder_cache(self.cache, idx)

class EncDecLM(nn.Module):
    """B: Transformer encoder-decoder. Training: each window is split at a random point E — the encoder reads tokens
    [0, E) bidirectionally, the decoder (starting from <|bos|>, cross-attending to the encoder) predicts tokens [E, T].
    At test time the prompt goes to the encoder and the continuation is decoded — exactly the task's shape."""
    is_encdec = True
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=2048, rope_base=10000,
                 enc_layers=None, enc_len_eval=768, bos_id=2, **_):
        super().__init__()
        ne = enc_layers or n_layer // 2; nd = n_layer - ne
        self.bos_id, self.enc_len_eval, self.ctx_len = bos_id, enc_len_eval, ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.enc = nn.ModuleList([EncoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(ne)])
        self.dec = nn.ModuleList([DecoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(nd)])
        self.enc_norm, self.norm_f = RMSNorm(d_model), RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _rope(self, start, T, device):
        p = torch.arange(start, start + T, device=device)[None]
        return self.rope_cos[p][:, None], self.rope_sin[p][:, None]
    def encode(self, idx, valid=None):
        cos, sin = self._rope(0, idx.size(1), idx.device)
        x = self.wte(idx); mask = None if valid is None else valid[:, None, None, :]
        for blk in self.enc: x = blk(x, cos, sin, mask)
        return self.enc_norm(x)
    def decode(self, idx, mem, mem_valid, start, cache=None, use_cache=False):
        cos, sin = self._rope(start, idx.size(1), idx.device)
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.dec):
            x, c = blk(x, cos, sin, mem, mem_valid, cache[i] if cache else None, use_cache); new.append(c)
        return self.lm_head(self.norm_f(x)), (new if use_cache else None)
    def forward(self, idx, targets=None, enc_len=None, **_):
        B, T = idx.shape
        E = int(enc_len or min(self.enc_len_eval, T - 1))
        mem = self.encode(idx[:, :E])
        dec_in = torch.cat([torch.full((B, 1), self.bos_id, dtype=torch.long, device=idx.device), idx[:, E:]], 1)
        logits, _ = self.decode(dec_in, mem, None, 0)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return EncDecRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ C: retention network (RetNet-style), D: hybrid
class Retention(nn.Module):
    """Multi-scale retention (Sun et al. 2023, "Retentive Network"). Attention without softmax:
         parallel form (training):   O = (Q Kᵀ ⊙ D) V,   D[n,m] = γ^(n−m) for n ≥ m, else 0
         recurrent form (generation): S_n = γ·S_{n−1} + k_nᵀ v_n,   o_n = q_n S_n
       Both give identical outputs. Each head has its own decay γ (short- to long-range memory), so generation needs
       only a fixed-size state per head — no growing KV-cache, O(1) cost per new token.
       Per-head normalisation + a SiLU output gate as in RetNet; RoPE on q/k for relative position."""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q, self.k = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.v, self.g = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
        self.register_buffer("log_gamma", torch.log(1 - 2.0 ** (-5 - torch.arange(h, dtype=torch.float32))), persistent=False)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = [m(x).view(B, T, self.h, self.hd).transpose(1, 2) for m in (self.q, self.k, self.v)]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if valid is not None:                                   # left padding: padded positions contribute nothing
            vm = valid[:, None, :, None].to(k.dtype); k, v = k * vm, v * vm
        q = q * self.hd ** -0.5
        lg = self.log_gamma.view(1, self.h, 1, 1)
        new_state = None
        if state is None:                                       # parallel form
            n = torch.arange(T, device=x.device, dtype=torch.float32)
            diff = n[:, None] - n[None, :]
            D = torch.where(diff >= 0, torch.exp(diff * lg), torch.zeros((), device=x.device)).to(q.dtype)
            o = ((q @ k.transpose(-1, -2)) * D) @ v
            if use_cache:
                w = torch.exp((T - 1 - n).view(1, 1, T, 1) * lg)                       # γ^(T−1−m)
                new_state = (k.float() * w).transpose(-1, -2) @ v.float()            # (B, H, hd, hd)
        else:                                                   # recurrent form, one token at a time
            new_state = state * torch.exp(lg) + k.float().transpose(-1, -2) @ v.float()
            o = (q.float() @ new_state).to(q.dtype)
        o = F.layer_norm(o.float(), (self.hd,)).to(x.dtype)     # per-head normalisation (GroupNorm, no affine)
        o = o.transpose(1, 2).reshape(B, T, C)
        return self.o(F.silu(self.g(x)) * o), (new_state if use_cache else None)

class RetBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.ret, self.mlp = Retention(d, h), SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        r, s = self.ret(self.n1(x), cos, sin, state, valid, use_cache)
        x = x + r
        return x + self.mlp(self.n2(x)), s

class RetNetLM(nn.Module):
    """C (attn_every=0): every block is retention.  D (attn_every=k): every k-th block is full softmax attention
    (ModernBlock), the rest retention — retention supplies cheap long-range decayed memory, the few attention layers
    give exact recall of specific earlier tokens (names, section numbers)."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=1792, rope_base=10000,
                 attn_every=0, n_kv_head=None, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.kinds = ["attn" if attn_every and (i + 1) % attn_every == 0 else "ret" for i in range(n_layer)]
        self.blocks = nn.ModuleList([
            ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, "rms", "swiglu", True) if kd == "attn"
            else RetBlock(d_model, n_head, ffn_hidden, dropout) for kd in self.kinds])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        valid = mask[:, 0, -1, :] if (mask is not None and cache is None) else None   # key validity at prefill
        x = self.wte(idx); new = []
        for i, (kd, blk) in enumerate(zip(self.kinds, self.blocks)):
            c = cache[i] if cache is not None else None
            if kd == "attn": x, c = blk(x, cos, sin, c, mask, use_cache)
            else:            x, c = blk(x, cos, sin, c, valid, use_cache)
            new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

# ------------------------------------------------------------------ E: prefix-LM (encoder/hybrid adapted for LM)
class PrefixLM(ModernLM):
    """E: the A1 stack with a different ATTENTION MASK. Prompt (prefix) tokens attend to the whole prompt in both
    directions, like a BERT encoder; continuation tokens attend causally, like GPT. One shared stack plays both roles.
    Training: split each window at a random E, loss only on the continuation (predictions from position E−1 on).
    Generation: prefill the prompt with the bidirectional mask, then ordinary cached causal decoding."""
    def __init__(self, *a, enc_len_eval=768, **kw):
        super().__init__(*a, **kw); self.enc_len_eval = enc_len_eval
    def forward(self, idx, targets=None, enc_len=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        if targets is None:                                       # generation path (mask supplied by the runner)
            return super().forward(idx, None, pos, mask, cache, use_cache, last_only)
        T = idx.size(1); E = int(enc_len or min(self.enc_len_eval, T - 1))
        i = torch.arange(T, device=idx.device)
        allowed = (i[None, :] <= i[:, None]) | ((i[:, None] < E) & (i[None, :] < E))   # causal ∪ (prefix × prefix)
        logits, _, _ = super().forward(idx, None, None, allowed[None, None], None, False, False)
        loss = F.cross_entropy(logits[:, E - 1:].float().reshape(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return PrefixRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ F: efficient local-global decoder
def alibi_slopes(n):
    """ALiBi head slopes (Press et al. 2022): geometric sequence; extra heads interleaved when n is not a power of 2"""
    def pow2(m): return [2 ** (-8 * (k + 1) / m) for k in range(m)]
    if math.log2(n).is_integer(): return pow2(n)
    p = 2 ** math.floor(math.log2(n))
    return pow2(p) + pow2(2 * p)[0::2][: n - p]

class LocalGlobalAttention(nn.Module):
    """Grouped-query attention (h query heads share kv key/value heads) with an ALiBi distance penalty added to the
    attention logits (no position embeddings at all). window=W makes it sliding-window attention: a query only sees
    the previous W tokens; window=None is full causal attention."""
    def __init__(self, d, h, kv, window):
        super().__init__()
        self.h, self.kv, self.hd, self.window = h, kv, d // h, window
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, bias, past=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class EffBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, window):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn, self.mlp = LocalGlobalAttention(d, h, kv, window), SwiGLU(d, hidden, dropout)
    def forward(self, x, bias, past=None, use_cache=False):
        a, present = self.attn(self.n1(x), bias, past, use_cache)
        x = x + a
        return x + self.mlp(self.n2(x)), present

class EfficientLM(nn.Module):
    """F: lightweight decoder. Sliding-window attention (window W) in most layers, full causal attention in every
    k-th layer, grouped-query attention (fewer key/value heads → smaller KV-cache), ALiBi instead of position
    embeddings. Local layers model nearby text; the few global layers can still reach any earlier token."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=4, ffn_hidden=2304,
                 window=256, global_every=4, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        self.windows = [None if global_every and (i + 1) % global_every == 0 else window for i in range(n_layer)]
        self.blocks = nn.ModuleList([EffBlock(d_model, n_head, n_kv_head, ffn_hidden, dropout, w) for w in self.windows])
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head)), persistent=False)
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _bias(self, qpos, kpos, key_ok, window):
        """(B|1, H, Tq, Tk) additive mask: −slope·distance for allowed pairs, −inf elsewhere"""
        dist = (qpos[:, :, None] - kpos[:, None, :]).float()               # (B|1, Tq, Tk)
        ok = (dist >= 0) & key_ok[:, None, :]
        if window is not None: ok = ok & (dist < window)
        b = -self.slopes.view(1, -1, 1, 1) * dist[:, None]
        return b.masked_fill(~ok[:, None], float("-inf"))
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if mask is None:                                          # training / plain forward: positions 0..T-1
            qpos = kpos = torch.arange(T, device=idx.device)[None]
            key_ok = torch.ones((1, T), dtype=torch.bool, device=idx.device)
        else:                                                     # generation: derive key positions from the mask
            key_ok = mask[:, 0, -1, :]                            # validity of every key (past + current)
            kpos = key_ok.long().cumsum(-1) - 1
            qpos = pos
        biases = {w: self._bias(qpos, kpos, key_ok, w) for w in set(self.windows)}
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.blocks):
            x, c = blk(x, biases[self.windows[i]], cache[i] if cache is not None else None, use_cache); new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

MODEL_REGISTRY = {"gpt2": GPT2LM, "modern": ModernLM, "encdec": EncDecLM, "retnet": RetNetLM, "hybrid": RetNetLM,
                  "prefixlm": PrefixLM, "efficient": EfficientLM}

def build_model(cfg, vocab):
    return MODEL_REGISTRY[cfg["arch"]](vocab=vocab, **cfg)

def count_params(m):
    return sum(p.numel() for p in m.parameters())      # tied weights counted once

In [5]:
# (backbone + adaptation methods copied unchanged from P3_02)
class Backbone(nn.Module):
    def __init__(self, lm, arch, attn):
        super().__init__()
        self.lm, self.arch = lm, arch
        self.bidir = attn == "bidir" or arch == "encdec"
        self.d = lm.wte.weight.shape[1]
        if arch == "encdec": lm.dec = nn.ModuleList()
        self.ctx_limit = lm.ctx_len if arch == "gpt2" else (2 * lm.ctx_len if arch in ("modern", "prefixlm", "encdec") else 10 ** 9)
    def blocks(self):
        return self.lm.enc if self.arch == "encdec" else self.lm.blocks
    def forward(self, idx, valid, prefix=None):
        lm, (B, T), dev = self.lm, idx.shape, idx.device
        P = 0 if prefix is None else prefix.size(0)
        x = lm.wte(idx)
        if P:
            x = torch.cat([prefix[None].expand(B, -1, -1).to(x.dtype), x], 1)
            valid = torch.cat([torch.ones(B, P, dtype=torch.bool, device=dev), valid], 1)
        L = T + P
        assert L <= self.ctx_limit, f"window {L} longer than the model's position range {self.ctx_limit}"
        pos = torch.arange(L, device=dev)[None]
        mask = valid[:, None, None, :] if self.bidir else None
        if self.arch == "encdec":
            cos, sin = lm._rope(0, L, dev)
            for blk in lm.enc: x = blk(x, cos, sin, mask)
            h = lm.enc_norm(x)
        elif self.arch == "gpt2":
            x = lm.drop(x + lm.wpe(pos))
            for blk in lm.blocks: x, _ = blk(x, None, mask, False)
            h = lm.ln_f(x)
        elif self.arch in ("modern", "prefixlm"):
            if lm.wpe is not None: x = x + lm.wpe(pos)
            cos, sin = lm.rope_cos[pos][:, None], lm.rope_sin[pos][:, None]
            x = lm.drop(x)
            for blk in lm.blocks: x, _ = blk(x, cos, sin, None, mask, False)
            h = lm.norm_f(x)
        elif self.arch == "efficient":
            dist = (pos[:, :, None] - pos[:, None, :]).float()
            biases = {}
            for w in set(lm.windows):
                if self.bidir:
                    ok = valid[:, None, :] & ((dist.abs() < w) if w is not None else torch.ones_like(dist, dtype=torch.bool))
                    ok = ok | torch.eye(L, dtype=torch.bool, device=dev)[None]
                    b = -lm.slopes.view(1, -1, 1, 1) * dist.abs()[:, None]
                else:
                    ok = dist >= 0
                    if w is not None: ok = ok & (dist < w)
                    b = -lm.slopes.view(1, -1, 1, 1) * dist[:, None]
                biases[w] = b.masked_fill(~ok[:, None], float("-inf"))
            for i, blk in enumerate(lm.blocks): x, _ = blk(x, biases[lm.windows[i]], None, False)
            h = lm.norm_f(x)
        else:
            raise ValueError(self.arch)
        return h[:, P:]

class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.scale = base, alpha / (math.sqrt(r) if rs else r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

_PERM_IDX, _PERM_LOCK = {}, threading.Lock()
def perm_index(d, device):
    key = (d, str(device))
    with _PERM_LOCK:
        if key not in _PERM_IDX:
            i = torch.arange(d, device=device); _PERM_IDX[key] = (i[None, :] - i[:, None]) % d
        return _PERM_IDX[key]

class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        d = base.out_features; self.base, self.variant = base, variant
        if variant == "plus":
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B = nn.Parameter(torch.zeros(d, r))
        else:
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.empty(d, r); nn.init.kaiming_uniform_(B, a=math.sqrt(5)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
    def forward(self, x):
        y = self.base(x)
        if self.variant == "plus":
            return y + self.scale * F.linear(F.linear(y, self.A), self.B)
        M = self.B @ self.A
        return F.linear(y, M.gather(1, perm_index(M.size(0), M.device)))

class IA3Linear(nn.Module):
    """(IA)^3: mode "out" rescales the layer's outputs (keys/values; `mask` limits it to part of a fused q|k|v
    output), mode "in" rescales its inputs (= the FFN hidden activation when wrapped around the down-projection)"""
    def __init__(self, base, mode, mask=None):
        super().__init__()
        self.base, self.mode = base, mode
        n = base.out_features if mode == "out" else base.in_features
        self.l = nn.Parameter(torch.ones(n))
        self.register_buffer("mask", mask if mask is not None else torch.ones(n, dtype=torch.bool), persistent=False)
    def forward(self, x):
        if self.mode == "in": return self.base(x * self.l.to(x.dtype))
        s = torch.where(self.mask, self.l, torch.ones_like(self.l))
        return self.base(x) * s.to(x.dtype)

class BiasLinear(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x): return self.base(x) + self.bias

class AdapterLinear(nn.Module):
    def __init__(self, base, dim):
        super().__init__()
        d = base.out_features; self.base = base
        self.down, self.up = nn.Linear(d, dim), nn.Linear(dim, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x); return y + self.up(F.gelu(self.down(y)))

ATTN_TYPES = (CausalSelfAttention, ModernAttention, LocalGlobalAttention)
MLP_TYPES = (MLP, SwiGLU, GeluMLP)

def target_linears(backbone, which):
    out = []
    for parent in list(backbone.blocks().modules()):
        kind = "attn" if isinstance(parent, ATTN_TYPES) else "mlp" if isinstance(parent, MLP_TYPES) else None
        if kind is None or (which == "attn" and kind != "attn"): continue
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear):
                out.append((parent, name, child, bool(getattr(child, "RESIDUAL_SCALE", False))))
    return out

def apply_method(backbone, cfg):
    m = cfg["method"]
    for p in backbone.parameters(): p.requires_grad = (m == "full")
    if m in ("full", "frozen", "prompt"): return
    if m == "topk":
        for blk in list(backbone.blocks())[-cfg["topk"]:]:
            for p in blk.parameters(): p.requires_grad = True
        return
    if m == "ia3":
        for parent in list(backbone.blocks().modules()):
            if isinstance(parent, CausalSelfAttention):                      # fused q|k|v: rescale only k and v
                d = parent.qkv.in_features
                parent.qkv = IA3Linear(parent.qkv, "out", torch.arange(3 * d) >= d)
            elif isinstance(parent, (ModernAttention, LocalGlobalAttention)):
                parent.kvp = IA3Linear(parent.kvp, "out")
            elif isinstance(parent, SwiGLU): parent.w2 = IA3Linear(parent.w2, "in")
            elif isinstance(parent, (MLP, GeluMLP)): parent.proj = IA3Linear(parent.proj, "in")
        return
    which = cfg["lora_targets"] if m in ("lora", "rslora", "lorma") else "all"
    for parent, name, lin, is_out in target_linears(backbone, which):
        if m in ("lora", "rslora"): setattr(parent, name, LoRALinear(lin, cfg["lora_r"], cfg["lora_alpha"], rs=(m == "rslora")))
        elif m == "lorma": setattr(parent, name, LoRMALinear(lin, cfg["lora_r"], cfg["lora_alpha"], cfg["lorma_variant"]))
        elif m == "adapter":
            if is_out: setattr(parent, name, AdapterLinear(lin, cfg["adapter_dim"]))
        elif m == "bitfit":
            if lin.bias is not None: lin.bias.requires_grad = True
            else: setattr(parent, name, BiasLinear(lin))
    if m == "bitfit":
        for mod in backbone.modules():
            if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                for p in mod.parameters(): p.requires_grad = True

def set_backbone_dropout(backbone, p):
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p
        if isinstance(mod, (CausalSelfAttention, ModernAttention)): mod.dropout = p

_P1_CACHE, _P1_LOCK = {}, threading.Lock()
def load_phase1(key):
    with _P1_LOCK:
        if key not in _P1_CACHE:
            d = P1_DIRS[key]
            info = load_json(d / "run_info.json") or load_json(d / "summary.json") or {}
            pcfg = dict(info.get("cfg") or {}); assert pcfg, f"no run_info.json/summary.json next to {d}"
            sd = torch.load(d / "weights_final.pt", map_location="cpu", weights_only=False)
            _P1_CACHE[key] = (pcfg, {k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
        return _P1_CACHE[key]

### Phase-4 additions
* **Gradual unfreezing** (Howard & Ruder 2018, "layer freezing/unfreezing strategy"): training starts with only the
  head and the top block trainable; the next lower block is unfrozen at equal intervals until all blocks train.
  Lower layers (general language knowledge) change least and latest.
* **`keep_layers=k`**: keep only the first k blocks of a Phase-1 model — a genuinely smaller student for distillation.

In [6]:
def apply_method_p4(backbone, cfg):
    if cfg["method"] == "gradual":
        for p in backbone.parameters(): p.requires_grad = False       # the training loop unfreezes block by block
        return
    apply_method(backbone, cfg)

def gradual_unfreeze(backbone, n_open):
    blocks = list(backbone.blocks())
    for b_i, blk in enumerate(blocks):
        want = b_i >= len(blocks) - n_open
        for p in blk.parameters(): p.requires_grad = want

_P1_CACHE, _P1_LOCK = {}, threading.Lock()
def load_phase1(key):
    with _P1_LOCK:
        if key not in _P1_CACHE:
            d = P1_DIRS[key]; assert d is not None, f"Phase-1 weights for {key} not found"
            info = load_json(d / "run_info.json") or load_json(d / "summary.json") or {}
            pcfg = dict(info.get("cfg") or {}); assert pcfg
            sd = torch.load(d / "weights_final.pt", map_location="cpu", weights_only=False)
            _P1_CACHE[key] = (pcfg, {k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
        return _P1_CACHE[key]

def build_backbone(cfg):
    pcfg, sd = load_phase1(cfg["model"])
    torch.manual_seed(cfg["seed"])
    lm = MODEL_REGISTRY[pcfg["arch"]](vocab=sd["wte.weight"].shape[0], **{k: v for k, v in pcfg.items() if k != "vocab"})
    if cfg["init"] == "pretrained": lm.load_state_dict(sd)
    bb = Backbone(lm, pcfg["arch"], cfg["attn"])
    k = cfg.get("keep_layers")
    if k:
        if bb.arch == "encdec": lm.enc = nn.ModuleList(list(lm.enc)[:k])
        else:
            lm.blocks = nn.ModuleList(list(lm.blocks)[:k])
            if bb.arch == "efficient": lm.windows = lm.windows[:k]
    set_backbone_dropout(bb, cfg["backbone_dropout"])
    apply_method_p4(bb, cfg)
    return bb, pcfg

## 4. Document representations
Every row ends with one separator token (a fixed "summary" position; for causal models its state has read the
whole row). `L` = `seq_len`.

| rep | rows per document | tokens used |
|---|---|---|
| `head` | 1 | first L−1 |
| `middle` | 1 | L−1 around the middle |
| `tail` | 1 | last L−1 (just before the removed decision) |
| `headtail` | 1 | first `head_tokens` + separator + last L−`head_tokens`−2 |
| `chunks` | ≤ `n_chunks` | the last `n_chunks` × (`chunk_len`−1) tokens, cut into consecutive chunks (chronological order) |

In [7]:
def doc_rows(toks, cfg, rep=None, n_chunks=None):
    rep = rep or cfg["rep"]; L = cfg["seq_len"]; N = len(toks); sep = np.array([SEP_ID])
    if rep == "head":   return [np.concatenate([toks[:L - 1], sep])]
    if rep == "tail":   return [np.concatenate([toks[max(0, N - (L - 1)):], sep])]
    if rep == "middle":
        s = max(0, N // 2 - (L - 1) // 2); return [np.concatenate([toks[s:s + L - 1], sep])]
    if rep == "headtail":
        h = cfg["head_tokens"]
        if N <= L - 1: return [np.concatenate([toks, sep])]
        return [np.concatenate([toks[:h], sep, toks[N - (L - h - 2):], sep])]
    if rep == "chunks":
        C, K = cfg["chunk_len"] - 1, n_chunks or cfg["n_chunks"]; rows, end = [], N
        while len(rows) < K and end > 0:
            s = max(0, end - C); rows.append(np.concatenate([toks[s:end], sep])); end = s
        return rows[::-1] or [sep]
    raise ValueError(rep)

def collate(split, idxs, cfg, device, rep=None, n_chunks=None):
    rows, owner = [], []
    for b, i in enumerate(idxs):
        for r in doc_rows(split.tokens(i), cfg, rep, n_chunks): rows.append(r); owner.append(b)
    T = max(len(r) for r in rows)
    idx = np.full((len(rows), T), PAD_ID, np.int64); valid = np.zeros((len(rows), T), bool); last = np.zeros(len(rows), np.int64)
    for k, r in enumerate(rows): idx[k, :len(r)] = r; valid[k, :len(r)] = True; last[k] = len(r) - 1
    K = max(Counter(owner).values()); cidx = np.zeros((len(idxs), K), np.int64); cmask = np.zeros((len(idxs), K), bool)
    fill = Counter()
    for k, b in enumerate(owner): cidx[b, fill[b]] = k; cmask[b, fill[b]] = True; fill[b] += 1
    t = lambda a: torch.from_numpy(a).to(device, non_blocking=True)
    out = {"idx": t(idx), "valid": t(valid), "last": t(last), "cidx": t(cidx), "cmask": t(cmask)}
    if split.labels is not None: out["y"] = t(split.labels[np.asarray(idxs)])
    return out

## 5. Document model
```
rows ─► backbone ─► token vectors ─► token pooling (mean | end separator | attention) ─► one vector per row (chunk)
     ─► chunk pooling (attention | mean | BiGRU+attention over the document's chunks) ─► dropout ─► linear (2 classes)
```
Attention pooling: `α = softmax(w·h_i)`, `v = Σ α_i h_i` — the model learns which tokens / chunks matter.

In [8]:
class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__(); self.w = nn.Linear(d, 1)
    def forward(self, h, mask):                                       # h (B, T, d), mask (B, T)
        s = self.w(h).squeeze(-1).masked_fill(~mask, -1e4)
        a = torch.softmax(s, -1); return (a[..., None] * h).sum(1), a

class DocModel(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__()
        d = backbone.d; self.backbone, self.cfg = backbone, cfg
        self.tok_attn = AttnPool(d) if cfg["tok_pool"] == "attn" else None
        self.gru = nn.GRU(d, d // 2, batch_first=True, bidirectional=True) if cfg["chunk_pool"] == "gru" else None
        self.chunk_attn = AttnPool(d) if cfg["chunk_pool"] in ("attn", "gru") else None
        self.drop = nn.Dropout(cfg["head_dropout"]); self.cls = nn.Linear(d, 2)
    def row_vectors(self, batch):
        h = self.backbone(batch["idx"], batch["valid"]); m = batch["valid"]
        with torch.autocast(device_type=h.device.type, enabled=False):
            h = h.float()
            if self.cfg["tok_pool"] == "last": return h[torch.arange(h.size(0), device=h.device), batch["last"]]
            if self.tok_attn is not None: return self.tok_attn(h, m)[0]
            return (h * m[..., None]).sum(1) / m.sum(1, keepdim=True).clamp(min=1)
    def doc_vector(self, rv, batch):
        with torch.autocast(device_type=rv.device.type, enabled=False):
            c = rv[batch["cidx"]] * batch["cmask"][..., None]; m = batch["cmask"]
            if c.size(1) == 1: return c[:, 0]
            if self.gru is not None:
                c, _ = self.gru(c)
            if self.chunk_attn is not None: return self.chunk_attn(c, m)[0]
            return c.sum(1) / m.sum(1, keepdim=True).clamp(min=1)
    def forward(self, batch):
        v = self.doc_vector(self.row_vectors(batch), batch)
        with torch.autocast(device_type=v.device.type, enabled=False):
            return self.cls(self.drop(v.float()))

def build_model(cfg, device):
    bb, pcfg = build_backbone(cfg)
    return DocModel(bb, cfg).to(device), pcfg

def count_parameters(model):
    seen, total, train = set(), 0, 0
    for p in model.parameters():
        if id(p) in seen: continue
        seen.add(id(p)); total += p.numel(); train += p.numel() if p.requires_grad else 0
    return total, train

## 6. Metric, threshold calibration, FP/FN analysis
Official metric: macro-F1 over {0, 1}; precision / recall of class 1. The model outputs P(accepted); the decision
threshold is chosen on dev to maximise macro-F1 (training prior ≈ 41% accepted vs ≈ 50% in dev/test, so 0.5 is
not optimal). Both thresholds are reported.

In [9]:
def official(y, p):
    return {"macro_f1": float(f1_score(y, p, average="macro", labels=[0, 1], zero_division=0)),
            "precision": float(precision_score(y, p, average="binary", pos_label=1, zero_division=0)),
            "recall": float(recall_score(y, p, average="binary", pos_label=1, zero_division=0)),
            "accuracy": float(np.mean(np.asarray(y) == np.asarray(p)))}

def best_threshold(y, prob):
    grid = np.linspace(0.05, 0.95, 181); sc = [official(y, (prob >= t).astype(int))["macro_f1"] for t in grid]
    return float(grid[int(np.argmax(sc))]), float(max(sc))

def fp_fn_report(split, prob, thr):
    y = split.labels; p = (prob >= thr).astype(int); L = split.lengths
    kind = np.where((y == 1) & (p == 1), "TP", np.where((y == 0) & (p == 0), "TN", np.where(p == 1, "FP", "FN")))
    rep = {k: {"n": int((kind == k).sum()), "median_tokens": float(np.median(L[kind == k])) if (kind == k).any() else None,
               "mean_confidence": float(np.mean(np.abs(prob[kind == k] - 0.5) + 0.5)) if (kind == k).any() else None} for k in ("TP", "TN", "FP", "FN")}
    q = np.quantile(L, [0, .25, .5, .75, 1.0])
    rep["macro_f1_by_length_quartile"] = {f"Q{j+1} ({int(q[j])}-{int(q[j+1])} tok)": official(y[(L >= q[j]) & (L <= q[j + 1])], p[(L >= q[j]) & (L <= q[j + 1])])["macro_f1"] for j in range(4)}
    rep["confusion[[TN,FP],[FN,TP]]"] = confusion_matrix(y, p, labels=[0, 1]).tolist()
    return rep

def write_submission(ids, preds, d, name="submission.zip"):
    obj = [{"id": i, "pred": int(v)} for i, v in zip(ids, preds)]
    assert len(obj) == len(SPLITS["test"]) and {o["id"] for o in obj} == set(SPLITS["test"].ids)
    save_json(obj, d / f"{Path(name).stem}_prediction.json")
    with zipfile.ZipFile(d / name, "w", zipfile.ZIP_DEFLATED) as z: z.writestr("prediction.json", json.dumps(obj))

@torch.no_grad()
def predict(model, split, cfg, device, idxs=None, batch=None):
    model.eval(); idxs = np.arange(len(split)) if idxs is None else idxs; out = np.zeros(len(idxs), np.float32)
    use_amp = device.type == "cuda"; bs = batch or max(4, cfg["batch"] * 2)
    for i in range(0, len(idxs), bs):
        b = collate(split, idxs[i:i + bs], cfg, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(b)
        out[i:i + bs] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
    return out

## 7. Training (+ knowledge distillation), experiment runner
AdamW, warmup + linear decay, fp16 autocast, gradient accumulation (`batch` × `accum` documents per update), class-
weighted cross-entropy. The dev set is scored after every epoch; the best epoch's trainable weights are kept.
**KD**: the teacher's P(accepted) for every training document (computed once, with the teacher's own
representation) is the soft target: `α·CE + (1−α)·T²·KL(teacher_T ‖ student_T)`.

In [10]:
def subset_indices(split, n, seed):
    if not n or n >= len(split): return np.arange(len(split))
    rng = np.random.default_rng(seed); out = []
    for c in (0, 1):                                                 # stratified by label
        idx = np.where(split.labels == c)[0]; k = int(round(n * len(idx) / len(split)))
        out.append(rng.choice(idx, k, replace=False))
    return np.sort(np.concatenate(out))

def load_trained(name, device):
    tcfg = dict(DEFAULTS); tcfg.update(load_json(EXPS_DIR / name / "config.json") or load_json(find_artifact(f"p4/exps/{name}/config.json")) or {})
    w = find_artifact(f"p4/exps/{name}/trainable.pt")
    assert w is not None, f"weights p4/exps/{name}/trainable.pt not found — attach the run that trained {name}"
    m, _ = build_model(tcfg, device)
    sd = torch.load(w, map_location="cpu", weights_only=False)
    res = m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()}, strict=False)
    assert not res.unexpected_keys, res.unexpected_keys
    return m, tcfg

def train_model(cfg, device, log):
    split = SPLITS["train"]; tr_idx = subset_indices(split, cfg["max_train_docs"], cfg["seed"])
    soft = None; teacher_params = None
    if cfg["teacher"]:
        teacher, tcfg = load_trained(cfg["teacher"], device); teacher_params = count_parameters(teacher)[0]
        t0 = time.time(); p = predict(teacher, split, tcfg, device, tr_idx)
        soft = dict(zip(tr_idx.tolist(), p.tolist())); log(f"teacher {cfg['teacher']} ({teacher_params/1e6:.1f}M): soft targets for {len(tr_idx)} docs in {time.time()-t0:.0f}s")
        del teacher
        if device.type == "cuda": torch.cuda.empty_cache()
    model, pcfg = build_model(cfg, device); total, trainable = count_parameters(model)
    if cfg["method"] == "gradual":
        n_blocks = len(list(model.backbone.blocks())); gradual_unfreeze(model.backbone, 1)
        trainable = count_parameters(model)[1]
    hp = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    bp = [p for n, p in model.named_parameters() if n.startswith("backbone.") and (p.requires_grad or cfg["method"] == "gradual")]
    groups = [{"params": [p for p in hp if p.ndim >= 2], "lr": cfg["lr_head"], "weight_decay": cfg["weight_decay"]},
              {"params": [p for p in hp if p.ndim < 2], "lr": cfg["lr_head"], "weight_decay": 0.0}]
    if bp and cfg["lr"] > 0:
        groups += [{"params": [p for p in bp if p.ndim >= 2], "lr": cfg["lr"], "weight_decay": cfg["weight_decay"]},
                   {"params": [p for p in bp if p.ndim < 2], "lr": cfg["lr"], "weight_decay": 0.0}]
    opt = torch.optim.AdamW([g for g in groups if g["params"]], betas=(0.9, 0.98), eps=1e-8)
    for g in opt.param_groups: g["base_lr"] = g["lr"]
    upd_per_epoch = math.ceil(len(tr_idx) / (cfg["batch"] * cfg["accum"])); total_upd = max(1, upd_per_epoch * cfg["epochs"])
    warm = max(1, int(cfg["warmup"] * total_upd))
    use_amp = device.type == "cuda"; scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    y = split.labels[tr_idx]; cw = None
    if cfg["class_weight"] == "balanced":
        cnt = np.bincount(y, minlength=2).astype(float); cw = torch.tensor(len(y) / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
    log(f"model {cfg['model']} ({pcfg['arch']}{', ' + str(cfg['keep_layers']) + ' layers' if cfg['keep_layers'] else ''}) "
        f"attn={cfg['attn']} method={cfg['method']} rep={cfg['rep']}({cfg['seq_len'] if cfg['rep'] != 'chunks' else str(cfg['n_chunks']) + 'x' + str(cfg['chunk_len'])}) "
        f"pool={cfg['tok_pool']}/{cfg['chunk_pool']} | params {total/1e6:.1f}M, trainable {trainable/1e6:.3f}M ({100*trainable/total:.2f}%) | "
        f"{len(tr_idx)} train docs ({y.mean():.1%} accepted), {total_upd} updates")
    if use_amp: torch.cuda.reset_peak_memory_stats(device)
    dev = SPLITS["dev"]; rng = np.random.default_rng(cfg["seed"]); hist, best = [], (-1.0, None, 0, 0.5)
    upd, t0 = 0, time.time()
    for ep in range(1, cfg["epochs"] + 1):
        model.train(); order = rng.permutation(tr_idx); run, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i in range(0, len(order), cfg["batch"]):
            if cfg["method"] == "gradual":
                n_open = min(n_blocks, 1 + (upd * n_blocks) // max(1, int(0.6 * total_upd)))   # all blocks open after 60% of training
                gradual_unfreeze(model.backbone, n_open)
            part = order[i:i + cfg["batch"]]; b = collate(split, part, cfg, device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(b)
            logits = logits.float(); loss = F.cross_entropy(logits, b["y"], weight=cw)
            if soft is not None:
                T, a = cfg["kd_T"], cfg["kd_alpha"]
                pt = torch.tensor([soft[int(j)] for j in part], device=device).clamp(1e-4, 1 - 1e-4)
                tl = torch.stack([torch.log1p(-pt), torch.log(pt)], -1) / T
                kl = (F.softmax(tl, -1) * (F.log_softmax(tl, -1) - F.log_softmax(logits / T, -1))).sum(-1).mean()
                loss = a * loss + (1 - a) * T * T * kl
            scaler.scale(loss / cfg["accum"]).backward(); run += loss.item(); nb += 1
            if nb % cfg["accum"] == 0 or i + cfg["batch"] >= len(order):
                lr_mult = (upd + 1) / warm if upd < warm else max(0.0, (total_upd - upd) / max(1, total_upd - warm))
                for g in opt.param_groups: g["lr"] = g["base_lr"] * lr_mult
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"] if p.grad is not None], cfg["grad_clip"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); upd += 1
        prob = predict(model, dev, cfg, device); thr, tuned = best_threshold(dev.labels, prob)
        msg = {"epoch": ep, "loss": run / max(nb, 1), "min": (time.time() - t0) / 60,
               "dev_f1@0.5": official(dev.labels, (prob >= 0.5).astype(int))["macro_f1"], "dev_f1_tuned": tuned, "thr": thr}
        if tuned > best[0]:
            best = (tuned, {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters()
                            if p.requires_grad or not n.startswith("backbone.") or cfg["method"] in ("gradual", "full")}, ep, thr)
        hist.append(msg); log(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in msg.items()}))
    model.load_state_dict(best[1], strict=False)
    info = {"total_params": total, "trainable_params": trainable, "train_min": (time.time() - t0) / 60, "history": hist,
            "peak_mem_gb": torch.cuda.max_memory_allocated(device) / 1e9 if use_amp else None, "best_epoch": best[2],
            "backbone_arch": pcfg["arch"], "teacher_params": teacher_params, "train_docs": int(len(tr_idx))}
    return model, info, set(best[1])

def save_trainable(model, names, path):
    sd = {n: p.detach().half().cpu() if p.is_floating_point() else p.detach().cpu() for n, p in model.named_parameters() if n in names}
    torch.save(sd, str(path) + ".tmp"); os.replace(str(path) + ".tmp", path)

def finish(name, cfg, d, prob_dev, prob_test, info, log, extra=None):
    dev = SPLITS["dev"]; thr, tuned = best_threshold(dev.labels, prob_dev)
    at05 = official(dev.labels, (prob_dev >= 0.5).astype(int)); att = official(dev.labels, (prob_dev >= thr).astype(int))
    rep = fp_fn_report(dev, prob_dev, thr)
    save_json({"at_0.5": at05, "tuned": att, "threshold": thr, "fp_fn": rep}, d / "dev_metrics.json")
    np.save(d / "prob_dev.npy", prob_dev); np.save(d / "prob_test.npy", prob_test)
    write_submission(SPLITS["test"].ids, (prob_test >= thr).astype(int), d, "submission.zip")
    write_submission(SPLITS["test"].ids, (prob_test >= 0.5).astype(int), d, "submission_thr05.zip")
    row = {"name": name, "model": cfg["model"] + (f"[{cfg['keep_layers']}L]" if cfg.get("keep_layers") else ""), "arch": info.get("backbone_arch"),
           "attn": cfg["attn"], "init": cfg["init"],
           "method": cfg["method"] + (f"-{cfg['lorma_variant']}" if cfg["method"] == "lorma" else "") + (f"+KD({cfg['teacher']})" if cfg["teacher"] else ""),
           "rep": cfg["rep"] + (f"{cfg['seq_len']}" if cfg["rep"] != "chunks" else f"{cfg['n_chunks']}x{cfg['chunk_len']}"),
           "pool": f"{cfg['tok_pool']}/{cfg['chunk_pool']}", "train_docs": info.get("train_docs"),
           "dev_f1_tuned": att["macro_f1"], "dev_f1@0.5": at05["macro_f1"], "threshold": thr, "dev_P": att["precision"], "dev_R": att["recall"],
           "dev_acc": att["accuracy"], "FP": rep["FP"]["n"], "FN": rep["FN"]["n"],
           "total_params_M": info["total_params"] / 1e6, "trainable_params_M": info["trainable_params"] / 1e6,
           "trainable_%": 100 * info["trainable_params"] / info["total_params"], "teacher_params_M": (info.get("teacher_params") or 0) / 1e6,
           "peak_mem_GB": info.get("peak_mem_gb"), "train_min": info["train_min"], "infer_ms_per_doc": info.get("infer_ms_per_doc"),
           "best_epoch": info.get("best_epoch"), **(extra or {})}
    save_json({"row": row, "info": info, "cfg": cfg}, d / "result.json")
    log(f"DONE dev macro-F1 {att['macro_f1']:.4f} @thr {thr:.2f} (0.5: {at05['macro_f1']:.4f}) | FP {rep['FP']['n']} FN {rep['FN']['n']}")
    return row

def run_experiment(name, device):
    cfg = exp_cfg(name); d = EXPS_DIR / name; d.mkdir(parents=True, exist_ok=True)
    if (d / "result.json").exists(): print(f"[{name}] already done — skipped"); return
    logf = open(d / "log.txt", "a")
    def log(msg):
        line = f"[{name} | {device}] {msg}"; print(line, flush=True); logf.write(line + "\n"); logf.flush()
    save_json(cfg, d / "config.json")
    if cfg["hier_from"]:
        run_hierarchical(name, cfg, d, device, log); logf.close(); return
    model, info, names = train_model(cfg, device, log)
    save_trainable(model, names, d / "trainable.pt")
    t0 = time.time(); prob_dev = predict(model, SPLITS["dev"], cfg, device)
    info["infer_ms_per_doc"] = 1000 * (time.time() - t0) / len(SPLITS["dev"])
    prob_test = predict(model, SPLITS["test"], cfg, device)
    finish(name, cfg, d, prob_dev, prob_test, info, log)
    logf.close(); del model
    if device.type == "cuda": torch.cuda.empty_cache()

## 8. Hierarchical two-stage model (`hier_from=<finished experiment>`)
1. The finished experiment's fine-tuned encoder embeds the last `hier_chunks` chunks of every document (inference
   only — much cheaper than training through all chunks), using its own token pooling → (docs, K, d).
2. A small model reads the sequence of chunk vectors: BiGRU → attention pooling → linear (2 classes), trained for
   `hier_epochs` with class weights; best epoch and threshold chosen on dev.
This is the ILDC-style hierarchical approach and covers far more of the document than any single window.

In [11]:
@torch.no_grad()
def embed_chunks(enc, split, cfg, K, device, idxs=None, bs=32):
    enc.eval(); idxs = np.arange(len(split)) if idxs is None else idxs; d = enc.backbone.d
    E = np.zeros((len(idxs), K, d), np.float16); M = np.zeros((len(idxs), K), bool); use_amp = device.type == "cuda"
    docs_per = max(1, bs // K)
    for i in range(0, len(idxs), docs_per):
        part = idxs[i:i + docs_per]; b = collate(split, part, cfg, device, rep="chunks", n_chunks=K)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            rv = enc.row_vectors(b)
        c = (rv[b["cidx"]] * b["cmask"][..., None]).float().cpu().numpy(); m = b["cmask"].cpu().numpy()
        k = c.shape[1]; E[i:i + len(part), K - k:] = c; M[i:i + len(part), K - k:] = m      # right-aligned: last chunk at position K-1
    return E, M

class HierHead(nn.Module):
    def __init__(self, d, h=256):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.1))
        self.gru = nn.GRU(d, h, batch_first=True, bidirectional=True); self.att = AttnPool(2 * h)
        self.pos = nn.Parameter(torch.zeros(1, 64, d)); self.cls = nn.Linear(2 * h, 2); self.drop = nn.Dropout(0.2)
    def forward(self, x, m):
        x = self.inp(x + self.pos[:, -x.size(1):]); x, _ = self.gru(x)
        v, a = self.att(x, m); return self.cls(self.drop(v)), a

def run_hierarchical(name, cfg, d, device, log):
    enc, ecfg = load_trained(cfg["hier_from"], device); ecfg = dict(ecfg, chunk_len=ecfg["seq_len"] if ecfg["rep"] != "chunks" else ecfg["chunk_len"])
    K = cfg["hier_chunks"]; tr_idx = subset_indices(SPLITS["train"], cfg["hier_max_train_docs"], cfg["seed"])
    t0 = time.time()
    Etr, Mtr = embed_chunks(enc, SPLITS["train"], ecfg, K, device, tr_idx)
    Edv, Mdv = embed_chunks(enc, SPLITS["dev"], ecfg, K, device); Ete, Mte = embed_chunks(enc, SPLITS["test"], ecfg, K, device)
    emb_min = (time.time() - t0) / 60; log(f"embedded {K} chunks x {ecfg['chunk_len']} tokens for {len(tr_idx)}+{len(Edv)}+{len(Ete)} docs in {emb_min:.1f} min")
    enc_params = count_parameters(enc)[0]; del enc
    if device.type == "cuda": torch.cuda.empty_cache()
    torch.manual_seed(cfg["seed"]); head = HierHead(Etr.shape[-1]).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.01)
    ytr = SPLITS["train"].labels[tr_idx]; cnt = np.bincount(ytr, minlength=2).astype(float)
    cw = torch.tensor(len(ytr) / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
    T = lambda a: torch.from_numpy(np.ascontiguousarray(a)).to(device)
    def probs(E, M):
        head.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(E), 256): out.append(torch.softmax(head(T(E[i:i+256]).float(), T(M[i:i+256]))[0], -1)[:, 1].cpu().numpy())
        return np.concatenate(out)
    rng = np.random.default_rng(cfg["seed"]); best = (-1, None, 0); t1 = time.time()
    for ep in range(1, cfg["hier_epochs"] + 1):
        head.train(); order = rng.permutation(len(tr_idx))
        for i in range(0, len(order), 64):
            j = order[i:i + 64]; logits, _ = head(T(Etr[j]).float(), T(Mtr[j]))
            loss = F.cross_entropy(logits, T(ytr[j]), weight=cw); opt.zero_grad(); loss.backward(); opt.step()
        pdv = probs(Edv, Mdv); thr, tuned = best_threshold(SPLITS["dev"].labels, pdv)
        log(json.dumps({"epoch": ep, "dev_f1_tuned": round(tuned, 4), "thr": round(thr, 2)}))
        if tuned > best[0]: best = (tuned, {k: v.detach().clone() for k, v in head.state_dict().items()}, ep)
    head.load_state_dict(best[1])
    pdv, pte = probs(Edv, Mdv), probs(Ete, Mte)
    with torch.no_grad():   # where does the attention look? mean weight per chunk position (last = just before the decision)
        _, a = head(T(Edv[:512]).float(), T(Mdv[:512])); att = a.mean(0).cpu().numpy().round(3).tolist()
    info = {"total_params": enc_params + count_parameters(head)[0], "trainable_params": count_parameters(head)[0],
            "train_min": emb_min + (time.time() - t1) / 60, "embed_min": emb_min, "best_epoch": best[2], "train_docs": int(len(tr_idx)),
            "backbone_arch": ecfg["model"], "chunk_attention_first_to_last": att}
    cfg_row = dict(cfg, model=ecfg["model"], attn=ecfg["attn"], method=f"hier({cfg['hier_from']})", rep="chunks", n_chunks=K,
                   chunk_len=ecfg["chunk_len"], tok_pool=ecfg["tok_pool"], chunk_pool="gru")
    log(f"mean chunk attention (first -> last chunk): {att}")
    finish(name, cfg_row, d, pdv, pte, info, log)

## 9. Unit tests (CPU, seconds)

In [12]:
def _tests():
    cfg = dict(DEFAULTS, seq_len=16, head_tokens=4, chunk_len=8, n_chunks=3)
    toks = np.arange(100, 140)
    r = doc_rows(toks, cfg, "tail")[0]; assert len(r) == 16 and r[-1] == SEP_ID and (r[:-1] == toks[-15:]).all()
    r = doc_rows(toks, cfg, "head")[0]; assert (r[:-1] == toks[:15]).all()
    r = doc_rows(toks, cfg, "headtail")[0]; assert len(r) == 16 and (r[:4] == toks[:4]).all() and r[4] == SEP_ID and (r[5:-1] == toks[-10:]).all()
    r = doc_rows(toks, cfg, "middle")[0]; assert len(r) == 16
    rows = doc_rows(toks, cfg, "chunks"); assert len(rows) == 3 and all(len(x) == 8 for x in rows)
    assert (np.concatenate([x[:-1] for x in rows]) == toks[-21:]).all(), "chunks must be the last tokens in order"
    assert len(doc_rows(np.arange(5), cfg, "chunks")) == 1 and len(doc_rows(np.zeros(0, np.int64), cfg, "tail")[0]) == 1
    # pooling
    class _BB(nn.Module):
        def __init__(self): super().__init__(); self.d = 2
        def forward(self, idx, valid): return torch.stack([idx.float(), -idx.float()], -1)
    for tp in ("mean", "last"):
        m = DocModel.__new__(DocModel); nn.Module.__init__(m); m.backbone, m.cfg, m.tok_attn = _BB(), dict(DEFAULTS, tok_pool=tp), None
        b = {"idx": torch.tensor([[2, 4, 6, 1]]), "valid": torch.tensor([[True, True, True, False]]), "last": torch.tensor([2])}
        v = m.row_vectors(b); assert abs(v[0, 0].item() - (4.0 if tp == "mean" else 6.0)) < 1e-6, (tp, v)
    # gradual unfreezing schedule opens blocks from the top
    class _Stack(nn.Module):
        def __init__(self): super().__init__(); self.bl = nn.ModuleList([nn.Linear(2, 2) for _ in range(4)])
        def blocks(self): return self.bl
    st = _Stack(); gradual_unfreeze(st, 1); assert [b.weight.requires_grad for b in st.bl] == [False, False, False, True]
    gradual_unfreeze(st, 3); assert [b.weight.requires_grad for b in st.bl] == [False, True, True, True]
    base = nn.Linear(16, 24, bias=False); x = torch.randn(5, 16)
    for mod in (LoRMALinear(base, 4, 8), LoRMALinear(base, 4, 8, "pi"), AdapterLinear(base, 6), BiasLinear(base)):
        assert torch.allclose(mod(x), base(x), atol=1e-5)
    y = np.array([0, 0, 1, 1]); assert official(y, y)["macro_f1"] == 1.0 and abs(official(y, np.array([0, 1, 1, 1]))["macro_f1"] - 0.7333) < 1e-3
    print("unit tests passed")
_tests()

unit tests passed


## 10. Run the queue (one worker thread per GPU)

In [13]:
def run_queue(names):
    q = _queue.Queue()
    for n in names: q.put(n)
    devices = [torch.device(f"cuda:{i}") for i in range(NUM_GPUS)] or [torch.device("cpu")]
    def worker(dev):
        while True:
            try: name = q.get_nowait()
            except _queue.Empty: return
            if time_left_h() < MIN_HOURS_PER_EXP: print(f"[{name}] skipped: {time_left_h():.2f} h left"); continue
            try:
                if dev.type == "cuda":
                    with torch.cuda.device(dev): run_experiment(name, dev)
                else: run_experiment(name, dev)
            except Exception:
                print(f"[{name}] FAILED:\n{traceback.format_exc()}", flush=True)
                (EXPS_DIR / name).mkdir(parents=True, exist_ok=True); (EXPS_DIR / name / "error.txt").write_text(traceback.format_exc())
    threads = [threading.Thread(target=worker, args=(dv,)) for dv in devices]
    for t in threads: t.start()
    for t in threads: t.join()

t_queue = time.time(); run_queue(RUN_QUEUE); print(f"queue finished in {(time.time() - t_queue)/60:.1f} min")

[s3_E_frozen | cuda:0] model E (prefixlm) attn=bidir method=frozen rep=tail(512) pool=mean/attn | params 110.1M, trainable 0.002M (0.00%) | 31652 train docs (41.4% accepted), 5937 updates
[s3_E_top4 | cuda:1] model E (prefixlm) attn=bidir method=topk rep=tail(512) pool=mean/attn | params 110.1M, trainable 28.320M (25.72%) | 31652 train docs (41.4% accepted), 3958 updates
[s3_E_frozen | cuda:0] {"epoch": 1, "loss": 0.7052, "min": 6.0314, "dev_f1@0.5": 0.5917, "dev_f1_tuned": 0.6238, "thr": 0.6}
[s3_E_top4 | cuda:1] {"epoch": 1, "loss": 0.611, "min": 10.2651, "dev_f1@0.5": 0.716, "dev_f1_tuned": 0.7324, "thr": 0.355}
[s3_E_frozen | cuda:0] {"epoch": 2, "loss": 0.6812, "min": 11.9938, "dev_f1@0.5": 0.5798, "dev_f1_tuned": 0.6159, "thr": 0.4}
[s3_E_frozen | cuda:0] {"epoch": 3, "loss": 0.6644, "min": 17.9448, "dev_f1@0.5": 0.5886, "dev_f1_tuned": 0.636, "thr": 0.45}
[s3_E_frozen | cuda:0] DONE dev macro-F1 0.6360 @thr 0.45 (0.5: 0.5886) | FP 136 FN 223
[s3_E_gradual | cuda:0] model E (pref

## 11. Results (this run + attached earlier runs)
`dev_f1_tuned` = official macro-F1 on dev with the dev-calibrated threshold (the threshold used for `submission.zip`);
`dev_f1@0.5` = with the plain 0.5 threshold (`submission_thr05.zip`).

In [14]:
rows = [load_json(f)["row"] for f in sorted(EXPS_DIR.glob("*/result.json"))]
if rows:
    df = pd.DataFrame(rows).sort_values("dev_f1_tuned", ascending=False)
    df.to_csv(ROOT / "p4" / "p4_experiments.csv", index=False)
    cols = ["name", "model", "attn", "method", "rep", "pool", "train_docs", "dev_f1_tuned", "dev_f1@0.5", "threshold", "dev_P", "dev_R",
            "FP", "FN", "trainable_params_M", "trainable_%", "peak_mem_GB", "train_min", "infer_ms_per_doc", "best_epoch"]
    display(df[[c for c in cols if c in df]].round(4).to_string(index=False))
    best = df.iloc[0]["name"]; m = load_json(EXPS_DIR / best / "dev_metrics.json")
    print(f"\nbest: {best} -> {ROOT}/p4/exps/{best}/submission.zip")
    print("FP/FN analysis of the best:", json.dumps(m["fp_fn"], indent=1))
for f in sorted(EXPS_DIR.glob("*/error.txt")): print(f"FAILED: {f.parent.name}\n{f.read_text()[-1500:]}")

'         name model   attn     method     rep      pool  train_docs  dev_f1_tuned  dev_f1@0.5  threshold  dev_P  dev_R  FP  FN  trainable_params_M  trainable_%  peak_mem_GB  train_min  infer_ms_per_doc  best_epoch\n         s2_E     E  bidir       full tail512 mean/attn       31652        0.7508      0.7122      0.330 0.7273 0.8048 150  97            110.1220     100.0000       4.2712    38.7536           10.1501           1\n   s3_E_lorma     E  bidir lorma-plus tail512 mean/attn       31652        0.7485      0.7178      0.370 0.7436 0.7586 130 120              3.0497       2.6948       2.9431    56.1445           12.6137           2\n    s3_E_top4     E  bidir       topk tail512 mean/attn       31652        0.7450      0.7370      0.450 0.8090 0.6479  76 175             28.3200      25.7169       1.6670    20.5197            9.9899           2\n s2_A0_causal    A0 causal       full tail512 last/attn       31652        0.7383      0.7159      0.280 0.7266 0.7646 143 117            1


best: s2_E -> /kaggle/working/cs781_p1/p4/exps/s2_E/submission.zip
FP/FN analysis of the best: {
 "TP": {
  "n": 400,
  "median_tokens": 3123.5,
  "mean_confidence": 0.7047688364982605
 },
 "TN": {
  "n": 347,
  "median_tokens": 3356.0,
  "mean_confidence": 0.8436580300331116
 },
 "FP": {
  "n": 150,
  "median_tokens": 4020.0,
  "mean_confidence": 0.6039652228355408
 },
 "FN": {
  "n": 97,
  "median_tokens": 3846.0,
  "mean_confidence": 0.7934526801109314
 },
 "macro_f1_by_length_quartile": {
  "Q1 (186-2152 tok)": 0.7951494010793734,
  "Q2 (2152-3393 tok)": 0.7822014051522248,
  "Q3 (3393-5440 tok)": 0.7338536585365854,
  "Q4 (5440-53090 tok)": 0.6894809296299296
 },
 "confusion[[TN,FP],[FN,TP]]": [
  [
   347,
   150
  ],
  [
   97,
   400
  ]
 ]
}
